# Baselines
- Naive Baseline: Predict tomorrow's returns will be the same as today's.
- Rolling Mean Baseline: Predict tomorrow's returns will be the mean of the last 30 days' returns.

## Data Loading and Preprocessing

In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import sys
import numpy as np
sys.path.append("..")
from src.data import load_data, compute_returns, create_sequences, scale_data

In [ ]:
df = load_data()
df_returns = df.copy()
df_returns = compute_returns(df_returns)
df_returns = df_returns.dropna()

# Scale returns to have mean 0 and standard deviation 1
stock_cols = ['AMZN', 'DPZ', 'BTC', 'NFLX']
scaled_data = scale_data(df_returns[stock_cols])

# Split time series data into training samples
X, y = create_sequences(scaled_data, window_size=60)


## Naive  & Rolling Mean Baselines
For the naive baseline, we predict $r_{t+1}​=0$ where $r_t$ is the returns on a stock at time t. In other words, we predict tomorrow's returns will be the same as today's. 

For the rolling mean baseline, we predict $r_{t+1} = 1/30 \sum_{i=t-29}^{t} r_i$. In other words, we predict tomorrow's returns to be the mean of the last 30 returns. 

In [31]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

stock_cols = ['AMZN','DPZ','BTC','NFLX']

y_pred_rolling = X.mean(axis=1)
y_pred_naive = X[:, -1, :] # Last day in each window
mse_naive    = mean_squared_error(y, y_pred_naive,    multioutput='raw_values')
mse_rolling  = mean_squared_error(y, y_pred_rolling,  multioutput='raw_values')

rmse_naive   = np.sqrt(mse_naive)
rmse_rolling = np.sqrt(mse_rolling)

results = pd.DataFrame({
    'MSE (Naive)': mse_naive,
    'RMSE (Naive)': rmse_naive,
    'MSE (Rolling)': mse_rolling,
    'RMSE (Rolling)': rmse_rolling
}, index=stock_cols)

# display nicely in notebook
display(results.style.format("{:.4f}"))

# also print summary scalars if you want
print("Overall (mean) MSE - Naive:", mse_naive.mean())
print("Overall (mean) MSE - Rolling:", mse_rolling.mean())
print("Overall (mean) RMSE - Naive:", rmse_naive.mean())
print("Overall (mean) RMSE - Rolling:", rmse_rolling.mean())

,MSE (Naive),RMSE (Naive),MSE (Rolling),RMSE (Rolling)
AMZN,2.0356,1.4268,1.0385,1.0191
DPZ,2.0535,1.4330,1.0182,1.0091
BTC,2.1819,1.4771,1.0497,1.0245
NFLX,1.9597,1.3999,1.0155,1.0077


Overall (mean) MSE - Naive: 2.0576780183702494
Overall (mean) MSE - Rolling: 1.0304887833241836
Overall (mean) RMSE - Naive: 1.4341928406128397
Overall (mean) RMSE - Rolling: 1.0151058150413328


## Discussion
The average naive MSE of ~2.06 tells us that, on average, the one-step naive forecast is 2.07 squared standard deviations from the true scaled return. 
The average naive RMSE of ~1.43 tells us that, on average, the one-step naive forecast is 1.43 standard deviations (not squared) from the true scaled return. 
The average rolling mean MSE of ~1.03 tells us that, on average, the one-step rolling mean forecast is 1.03 squared standard deviations from the true scaled return. 
The average rolling mean RMSE of ~1.02 tells us that, on average, the one-step rolling mean forecast is 1.02 standard deviations (not squared) from the true scaled return.
### Naive Basline
The MSE and RMSE figures here are, unsurprisingly, large. 
We can see that the naive baseline performs similarly across each stock, though the losses are slightly lower for the Netflix stock. This could be due to the NFLX stock's lag-1 autocorrelation being the largest, telling us that up days tend to follow up days and down follow down moreso for this stock than the others. This means our naive "predict tomorrow = today" baseline will be correct more often as today's returns are a decent proxy for tomorrow's. 
### Rolling Mean Baseline
This baseline performs significantly better than the naive one, with MSEs and RMSEs of around 1 across the board. This is expected, as our predictions with the rolling mean baseline take into account information from the previous 30 daily returns, not just the previous 1. Averaging over the previous 30 days gives us a lower variance by reducing noise, and better robustness to outliers as a single outlier will not move the 30-day average much. 

